## 7. Summary: What Sam Has vs What We Need

**Available masses**: 3.54, 4.13, 7.08 $\mu$eV (band edges + midpoint)

**Our band**: 856--1712 MHz $\Leftrightarrow$ 3.54--7.08 $\mu$eV (exact match!)

**Population models**:
- Young: $\tau_{\rm ohm} = 10$ Myr, 10 realizations, 3 masses
- Old: $\tau_{\rm ohm} = 1$ Tyr, 4 realizations, 2 masses (missing 7.08 $\mu$eV)

**To do**:
1. Sam needs to generate more masses (finer grid between 3.54 and 7.08 $\mu$eV)
2. Run `Analyze_Population.py` on any populations missing `Combined_Flux.dat`
3. Integrate `Mix_Match.py` output into the sideband analysis pipeline
4. Understand signal-to-noise: compare predicted flux vs our per-channel sensitivity

In [ ]:
# Import Sam's functions (adapted from Mix_Match.py)
sys.path.insert(0, BASE)
from Mix_Match import NFW, v0_NS, v0_DM, sample_NS_position, sample_theta, total_num_pulsars

# Quick test of Mix_Match: how many NSs in a population?
np.random.seed(42)
N_young = total_num_pulsars(young=True)
N_old = total_num_pulsars(young=False, tau_ohm=1e7)
print(f"Young population: {N_young} NSs (rate 5.4e-5/yr * 30 Myr)")
print(f"Old population (tau=10Myr): {N_old} NSs")
print(f"Total: {N_young + N_old}")
print(f"\nFor reference: Ntot = Poisson(N_young + N_old)")
print(f"Drawing Ntot = {np.random.poisson(lam=(N_young + N_old))}")

# Show what a resampled spatial distribution looks like
N_sample = 2000
positions_young = np.array([sample_NS_position(young=True) for _ in range(N_sample)])
positions_old = np.array([sample_NS_position(young=False) for _ in range(N_sample)])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Young
r_young = positions_young[:,2]  # kpc
axes[0].hist(r_young * 1e3, bins=50, color='cornflowerblue', alpha=0.7, label='Young')
axes[0].hist(positions_old[:,2] * 1e3, bins=50, color='maroon', alpha=0.5, label='Old')
axes[0].set_xlabel(r'$r_{\rm GC}$ [pc]')
axes[0].set_ylabel(r'Count')
axes[0].set_title(r'Galactocentric Distance (resampled)')
axes[0].legend()

# NFW profile check
r_vals = np.logspace(-2, 2, 100)  # kpc
rho_vals = [NFW(r) for r in r_vals]
axes[1].plot(r_vals, rho_vals, 'k-', lw=2, label=r'NFW ($\rho_0 = 0.346$ GeV/cm$^3$, $r_s = 20$ kpc)')
axes[1].set_xlabel(r'$r$ [kpc]')
axes[1].set_ylabel(r'$\rho_{\rm DM}$ [GeV/cm$^3$]')
axes[1].set_title(r'Dark Matter Density Profile')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].axvline(8.3, color='maroon', ls='--', label=r'$r_\oplus = 8.3$ kpc')
axes[1].legend()

plt.tight_layout()
plt.savefig('../plots/sam_resampled_spatial.png', dpi=150)
plt.show()

## 6. Mix & Match: Resampled Population

Run Sam's `Mix_Match.py` logic to generate a resampled population and see the resulting spectrum.

In [ ]:
# Compare young vs old populations at Ma=4.13e-6
h_eV = 4.1357e-15
MassA = 4.13e-6

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

for pop_tag, pop_dir, color, label in [
    ('Young', POP_YOUNG, 'cornflowerblue', r'Young ($\tau_{\rm ohm} = 10$ Myr)'),
    ('Old', POP_OLD, 'maroon', r'Old ($\tau_{\rm ohm} = 1$ Tyr)'),
]:
    flux_file = OUT_DIR + '/' + pop_dir + '/Ma_4.130e-06/Pop_0/Combined_Flux.dat'
    if not os.path.exists(flux_file):
        print(f"  {pop_tag}: no Combined_Flux.dat")
        continue
    fd = np.loadtxt(flux_file)
    freqs = fd[:,2] / h_eV / 1e6
    
    bins = np.linspace(freqs.min(), freqs.max(), 200)
    bw = (bins[1] - bins[0]) * 1e6
    spectrum, edges = np.histogram(freqs, bins=bins, weights=fd[:,1])
    spectrum /= bw
    centers = 0.5 * (edges[:-1] + edges[1:])
    
    n_ns = len(np.unique(fd[:,0]))
    total_flux = np.sum(fd[:,1]) / (MassA * 1e-5 / 6.58e-16)
    
    axes[0].plot(centers, spectrum * 1e6, color=color, lw=1,
                 label=label + r' (%d NSs, $\Sigma F \sim %.1e$ Jy)' % (n_ns, total_flux))
    
    # Load population properties
    pop_file = OUT_DIR + '/' + pop_dir + '/Population_Details_Pop_0_.txt'
    pop = np.loadtxt(pop_file)
    axes[1].hist(np.log10(pop[:,4]), bins=50, color=color, alpha=0.5, label=label + ' (%d NSs)' % len(pop))
    
    print(f"{pop_tag}: {n_ns} converting NSs out of {len(pop)} total, "
          f"total flux ~ {total_flux:.3e} Jy")

axes[0].set_xlabel(r'Frequency [MHz]')
axes[0].set_ylabel(r'Flux Density [$\mu$Jy] at $g = 10^{-12}$ GeV$^{-1}$')
axes[0].set_title(r'Young vs Old: Spectrum at $m_a = 4.13\;\mu$eV')
axes[0].legend(fontsize=13)

axes[1].set_xlabel(r'$\log_{10}(B_{\rm final}$ / G)')
axes[1].set_ylabel(r'Count')
axes[1].set_title(r'Magnetic Field Distributions')
axes[1].legend(fontsize=13)

plt.tight_layout()
plt.savefig('../plots/sam_young_vs_old.png', dpi=150)
plt.show()

## 5. Young vs Old Population Comparison

In [ ]:
# Compare spectra across the three available masses
h_eV = 4.1357e-15
masses = {'3.540e-06': 3.54e-6, '4.130e-06': 4.13e-6, '7.080e-06': 7.08e-6}
colors = {'3.540e-06': 'cornflowerblue', '4.130e-06': 'forestgreen', '7.080e-06': 'maroon'}

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

for mass_label, mass_val in masses.items():
    flux_file = OUT_DIR + '/' + POP_YOUNG + '/Ma_' + mass_label + '/Pop_0/Combined_Flux.dat'
    if not os.path.exists(flux_file):
        print(f"  {mass_label}: no Combined_Flux.dat, skipping")
        continue
    
    fd = np.loadtxt(flux_file)
    freqs = fd[:,2] / h_eV / 1e6  # MHz
    
    freq_center = mass_val / h_eV / 1e6
    bins = np.linspace(freqs.min(), freqs.max(), 200)
    bw = (bins[1] - bins[0]) * 1e6  # Hz
    
    spectrum, edges = np.histogram(freqs, bins=bins, weights=fd[:,1])
    spectrum /= bw  # Jy
    centers = 0.5 * (edges[:-1] + edges[1:])
    
    total_flux = np.sum(fd[:,1]) / (mass_val * 1e-5 / 6.58e-16)
    n_ns = len(np.unique(fd[:,0]))
    
    label = r'$m_a = %.2f\;\mu$eV ($\nu_0 = %.0f$ MHz, %d NSs)' % (mass_val*1e6, freq_center, n_ns)
    axes[0].plot(centers, spectrum * 1e6, color=colors[mass_label], lw=1, label=label)
    
    # Fractional frequency offset
    delta_f = (centers - freq_center) / freq_center * 1e3  # per mille
    axes[1].plot(delta_f, spectrum * 1e6, color=colors[mass_label], lw=1, label=label)
    
    print(f"  {mass_label}: {n_ns} NSs, {len(fd)} photons, total flux ~ {total_flux:.3e} Jy")

axes[0].set_xlabel(r'Frequency [MHz]')
axes[0].set_ylabel(r'Flux Density [$\mu$Jy] at $g = 10^{-12}$ GeV$^{-1}$')
axes[0].set_title(r'Young Pop 0: Spectra at Three Axion Masses')
axes[0].legend(fontsize=13)

axes[1].set_xlabel(r'$(\nu - \nu_0) / \nu_0$ [$\textperthousand$]')
axes[1].set_ylabel(r'Flux Density [$\mu$Jy]')
axes[1].set_title(r'Normalized Frequency Offset')
axes[1].legend(fontsize=13)

plt.tight_layout()
plt.savefig('../plots/sam_three_masses_comparison.png', dpi=150)
plt.show()

## 4. Compare All Three Masses

In [ ]:
# Per-NS total flux
ns_indices = np.unique(flux_data[:,0]).astype(int)
ns_total_flux = np.zeros(len(ns_indices))
for i, ns_idx in enumerate(ns_indices):
    mask = flux_data[:,0] == ns_idx
    ns_total_flux[i] = np.sum(flux_data[mask, 1])  # Jy-Hz

# Sort by flux
sort_idx = np.argsort(ns_total_flux)[::-1]
ns_indices_sorted = ns_indices[sort_idx]
ns_flux_sorted = ns_total_flux[sort_idx]

# Cumulative flux
cumflux = np.cumsum(ns_flux_sorted) / np.sum(ns_flux_sorted)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Flux distribution
axes[0].hist(np.log10(ns_total_flux[ns_total_flux > 0]), bins=50, color='cornflowerblue', alpha=0.8)
axes[0].set_xlabel(r'$\log_{10}$(Total flux weight / Jy-Hz)')
axes[0].set_ylabel(r'Count')
axes[0].set_title(r'Per-NS Flux Distribution')

# Cumulative
axes[1].plot(np.arange(1, len(cumflux)+1), cumflux * 100, color='forestgreen', lw=2)
axes[1].set_xlabel(r'Number of brightest NSs')
axes[1].set_ylabel(r'Cumulative flux [\%]')
axes[1].set_title(r'Cumulative Flux (brightest first)')
axes[1].axhline(90, color='maroon', ls='--', alpha=0.5)
axes[1].axhline(50, color='goldenrod', ls='--', alpha=0.5)
n_50 = np.searchsorted(cumflux, 0.5) + 1
n_90 = np.searchsorted(cumflux, 0.9) + 1
axes[1].text(n_50 * 1.5, 52, f'{n_50} NSs = 50\\%', fontsize=14)
axes[1].text(n_90 * 1.1, 92, f'{n_90} NSs = 90\\%', fontsize=14)

# Top 10 NS distances
top10 = ns_indices_sorted[:10]
for i, ns_idx in enumerate(top10):
    ns_info = pop_young_0[ns_idx]
    loc = ns_info[10:13]
    dist = np.sqrt(np.sum((loc - xE)**2))
    print(f"NS {ns_idx}: B={ns_info[4]:.2e} G, P={ns_info[5]:.4f} s, "
          f"age={ns_info[7]:.1f} Myr, dist={dist:.3f} kpc, "
          f"rho_DM={ns_info[8]:.2e} GeV/cm^3, flux={ns_total_flux[sort_idx[i]]:.3e} Jy-Hz")

# Top NS spatial positions
top_ns_locs = pop_young_0[ns_indices_sorted[:50], 10:13]
top_ns_dists = np.sqrt(np.sum((top_ns_locs - xE)**2, axis=1))
axes[2].scatter(top_ns_locs[:,0], top_ns_locs[:,1], 
                c=np.log10(ns_flux_sorted[:50]), cmap='hot', s=50, edgecolors='k', lw=0.5)
axes[2].plot(0, 8.3, 'c*', markersize=15, label=r'Earth')
axes[2].plot(0, 0, 'g+', markersize=15, label=r'GC')
axes[2].set_xlabel(r'$x$ [kpc]')
axes[2].set_ylabel(r'$y$ [kpc]')
axes[2].set_title(r'Top 50 brightest NSs')
axes[2].legend(fontsize=14)
axes[2].set_aspect('equal')

plt.tight_layout()
plt.savefig('../plots/sam_per_ns_flux.png', dpi=150)
plt.show()

## 3. Per-NS Flux Contributions: Which NSs dominate?

In [ ]:
# Spectrum: bin photon weights by energy to get flux density vs frequency
h_eV = 4.1357e-15  # eV*s
MassA = 4.13e-6

freqs_hz = flux_data[:,2] / h_eV
freqs_mhz = freqs_hz / 1e6

# Bin into frequency bins matching our MeerKAT channel width (~26 kHz)
chan_width_hz = 26123.05  # Hz
chan_width_mhz = chan_width_hz / 1e6

freq_min = freqs_mhz.min() - 0.01
freq_max = freqs_mhz.max() + 0.01
nbins = int((freq_max - freq_min) / chan_width_mhz)
bins = np.linspace(freq_min, freq_max, max(nbins, 200))
bin_width_hz = (bins[1] - bins[0]) * 1e6

# Flux density = sum(flux_weight) / bin_width  [Jy-Hz / Hz = Jy]
flux_density, bin_edges = np.histogram(freqs_mhz, bins=bins, weights=flux_data[:,1])
flux_density /= bin_width_hz  # Jy

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Full spectrum
axes[0].plot(bin_centers, flux_density * 1e6, color='cornflowerblue', lw=0.8)
axes[0].set_xlabel(r'Frequency [MHz]')
axes[0].set_ylabel(r'Flux Density [$\mu$Jy] at $g_{a\gamma\gamma} = 10^{-12}$ GeV$^{-1}$')
axes[0].set_title(r'Young Pop 0 Spectrum ($m_a = 4.13 \times 10^{-6}$ eV)')
axes[0].axvline(MassA / h_eV / 1e6, color='maroon', ls='--', alpha=0.5, label=r'$\nu = m_a / h$')
axes[0].legend()

# Zoom on core
core_mask = np.abs(bin_centers - np.median(freqs_mhz)) < 0.5
if np.any(core_mask):
    axes[1].plot(bin_centers[core_mask], flux_density[core_mask] * 1e6, color='cornflowerblue', lw=1)
    axes[1].set_xlabel(r'Frequency [MHz]')
    axes[1].set_ylabel(r'Flux Density [$\mu$Jy]')
    axes[1].set_title(r'Zoom: Central $\pm 0.5$ MHz')
    axes[1].axvline(MassA / h_eV / 1e6, color='maroon', ls='--', alpha=0.5, label=r'$\nu = m_a / h$')
    axes[1].legend()

plt.tight_layout()
plt.savefig('../plots/sam_flux_spectrum_ma4p13.png', dpi=150)
plt.show()

print(f"Peak flux density: {flux_density.max() * 1e6:.3f} uJy")
print(f"MeerKAT channel width: {chan_width_hz/1e3:.1f} kHz")

In [ ]:
# Load Combined_Flux.dat for young population, Ma=4.13e-6, Pop 0
flux_file = OUT_DIR + '/' + POP_YOUNG + '/Ma_4.130e-06/Pop_0/Combined_Flux.dat'
flux_data = np.loadtxt(flux_file)

# Columns from README/Analyze_Population.py:
# 0: NS_index, 1: flux_weight (Jy-Hz), 2: photon_energy (eV), 3: conversion_prob @ g=1e-12
# 4: x (kpc), 5: y (kpc), 6: z (kpc)

print(f"Combined flux data: {len(flux_data)} photon samples")
print(f"Unique NSs: {len(np.unique(flux_data[:,0]))}")
print(f"\nPhoton energies: {flux_data[:,2].min():.6e} -- {flux_data[:,2].max():.6e} eV")
print(f"  (axion mass = 4.130e-6 eV)")
print(f"  Fractional spread: {(flux_data[:,2].max() - flux_data[:,2].min()) / 4.13e-6 * 100:.2f}%")

# Convert energy to frequency: E = h*nu => nu = E/h
h_eV = 4.1357e-15  # eV*s
freqs_hz = flux_data[:,2] / h_eV
freqs_mhz = freqs_hz / 1e6
print(f"\nCorresponding frequencies: {freqs_mhz.min():.3f} -- {freqs_mhz.max():.3f} MHz")
print(f"  Center: {np.median(freqs_mhz):.3f} MHz")

print(f"\nFlux weights: {flux_data[:,1].min():.3e} -- {flux_data[:,1].max():.3e} Jy-Hz")
print(f"Conversion probs: {flux_data[:,3].min():.3e} -- {flux_data[:,3].max():.3e}")

# Total flux estimate (from Sam's code)
MassA = 4.13e-6
total_flux = np.sum(flux_data[:,1]) / (MassA * 1e-5 / 6.58e-16)
print(f"\nRough total flux @ g=1e-12: {total_flux:.3e} Jy")

## 2. Combined Flux: Spectral Structure at $m_a = 4.13 \times 10^{-6}$ eV

In [ ]:
# Plot population properties
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# B-field distribution
axes[0,0].hist(np.log10(pop_young_0[:,4]), bins=50, color='cornflowerblue', alpha=0.8)
axes[0,0].set_xlabel(r'$\log_{10}(B_{\rm final}$ / G)')
axes[0,0].set_ylabel(r'Count')
axes[0,0].set_title(r'Magnetic Field (evolved)')

# Period distribution
axes[0,1].hist(np.log10(pop_young_0[:,5]), bins=50, color='forestgreen', alpha=0.8)
axes[0,1].set_xlabel(r'$\log_{10}(P_{\rm final}$ / s)')
axes[0,1].set_ylabel(r'Count')
axes[0,1].set_title(r'Spin Period (evolved)')

# Age distribution
axes[0,2].hist(pop_young_0[:,7], bins=50, color='maroon', alpha=0.8)
axes[0,2].set_xlabel(r'Age [Myr]')
axes[0,2].set_ylabel(r'Count')
axes[0,2].set_title(r'Age Distribution')

# Spatial distribution (x-y plane)
axes[1,0].scatter(pop_young_0[:,10], pop_young_0[:,11], s=1, alpha=0.3, c='cornflowerblue')
axes[1,0].plot(0, 8.3, 'r*', markersize=15, label=r'Earth')
axes[1,0].plot(0, 0, 'k+', markersize=15, label=r'GC')
axes[1,0].set_xlabel(r'$x$ [kpc]')
axes[1,0].set_ylabel(r'$y$ [kpc]')
axes[1,0].set_title(r'Spatial Distribution ($x$-$y$)')
axes[1,0].legend(fontsize=14)
axes[1,0].set_aspect('equal')

# Distance from Earth
axes[1,1].hist(dists, bins=50, color='goldenrod', alpha=0.8)
axes[1,1].set_xlabel(r'Distance from Earth [kpc]')
axes[1,1].set_ylabel(r'Count')
axes[1,1].set_title(r'Distance Distribution')

# DM density vs distance from GC
r_gc = np.sqrt(pop_young_0[:,10]**2 + pop_young_0[:,11]**2 + pop_young_0[:,12]**2)
axes[1,2].scatter(r_gc, pop_young_0[:,8], s=1, alpha=0.3, c='firebrick')
axes[1,2].set_xlabel(r'$r_{\rm GC}$ [kpc]')
axes[1,2].set_ylabel(r'$\rho_{\rm DM}$ [GeV/cm$^3$]')
axes[1,2].set_title(r'DM Density vs GC Distance')
axes[1,2].set_yscale('log')
axes[1,2].set_xscale('log')

plt.suptitle(r'Young NS Population (Pop 0, $\tau_{\rm ohm} = 10$ Myr, %d NSs)' % len(pop_young_0), fontsize=18)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('../plots/sam_young_pop_properties.png', dpi=150)
plt.show()

In [ ]:
# Load young population 0
pop_young_0 = np.loadtxt(OUT_DIR + '/' + POP_YOUNG + '/Population_Details_Pop_0_.txt')

# Column definitions from Create_Population.py:
# 0: index, 1: B0 [G], 2: P0 [s], 3: chi0 [rad], 4: B_final [G], 5: P_final [s],
# 6: chi_final [rad], 7: age [Myr], 8: rho_DM [GeV/cm^3], 9: v0_DM [km/s],
# 10: x [kpc], 11: y [kpc], 12: z [kpc], 13: M_NS [Msun], 14: R_NS [km],
# 15: view_angle [rad], 16: doppler_shift

print(f"Young Pop 0: {len(pop_young_0)} neutron stars")
print(f"\nField strengths: B0 = {pop_young_0[:,1].min():.2e} -- {pop_young_0[:,1].max():.2e} G")
print(f"                 B_final = {pop_young_0[:,4].min():.2e} -- {pop_young_0[:,4].max():.2e} G")
print(f"Periods:         P0 = {pop_young_0[:,2].min():.4f} -- {pop_young_0[:,2].max():.4f} s")
print(f"                 P_final = {pop_young_0[:,5].min():.4f} -- {pop_young_0[:,5].max():.4f} s")
print(f"Ages:            {pop_young_0[:,7].min():.2f} -- {pop_young_0[:,7].max():.2f} Myr")
print(f"DM density:      {pop_young_0[:,8].min():.2e} -- {pop_young_0[:,8].max():.2e} GeV/cm^3")
print(f"Distances from Earth:")
xE = np.array([0.0, 8.3, 0.0])
dists = np.sqrt(np.sum((pop_young_0[:,10:13] - xE)**2, axis=1))
print(f"  min={dists.min():.3f} kpc, median={np.median(dists):.3f} kpc, max={dists.max():.3f} kpc")

## 1. Population Properties (Young NS, Pop 0)

In [ ]:
import os
os.environ["PATH"] = '/global/software/sl-7.x86_64/modules/tools/texlive/2016/bin/x86_64-linux/:' + os.environ["PATH"]

import sys
sys.path.insert(0, '/global/scratch/projects/pc_heptheory/jbenabou')
import plotting_defaults

import numpy as np
import matplotlib.pyplot as plt
import glob
import warnings

BASE = '/global/scratch/projects/pc_heptheory/jbenabou/NS_megaproject/MeerKAT_data/meerkat_reduction_project/reference/Real_Analysis/Real_Analysis'
OUT_DIR = BASE + '/Output_Files'

# Population directories
POP_YOUNG = 'PopYoung_WB_TauO_1.00e+07_B_12.79_sB_0.51_P_1.07_sP_14.56_'
POP_OLD = 'PopOld_WB_TauO_1.00e+12_B_12.82_sB_0.56_P_0.19_sP_22.19_'

# Available masses
young_masses = sorted(glob.glob(OUT_DIR + '/' + POP_YOUNG + '/Ma_*'))
old_masses = sorted(glob.glob(OUT_DIR + '/' + POP_OLD + '/Ma_*'))

print("Young population masses:")
for m in young_masses:
    print(f"  {os.path.basename(m)}")
print(f"\nOld population masses:")
for m in old_masses:
    print(f"  {os.path.basename(m)}")

# Count population realizations
young_pops = sorted(glob.glob(OUT_DIR + '/' + POP_YOUNG + '/Population_Details_Pop_*_.txt'))
old_pops = sorted(glob.glob(OUT_DIR + '/' + POP_OLD + '/Population_Details_Pop_*_.txt'))
print(f"\nYoung: {len(young_pops)} population realizations")
print(f"Old:   {len(old_pops)} population realizations")

# Sam Witte's NS Population Modeling Code: Exploration

**Source**: `Real_Analysis.zip` from Sam Witte (2026-04-15)

This notebook explores the pre-computed neutron star populations and their axion conversion signals.

## Data Structure

```
Output_Files/
  PopYoung_WB_.../ (tau_ohm=10 Myr, young NS population)
    Population_Details_Pop_N_.txt  -- NS properties for population realization N
    Ma_X.XXXe-06/                 -- axion mass directory
      Pop_N/                      -- population realization
        NS_i__Theta_X.XXX.txt     -- per-NS ray-tracing output (weight, energy, prob)
        Combined_Flux.dat         -- compiled flux from Analyze_Population.py
  PopOld_WB_.../ (tau_ohm=1 Tyr, old NS population)
    (same structure)
```

**Combined_Flux.dat columns**: `[NS_index, flux_weight (Jy-Hz), photon_energy (eV), conversion_prob @ g=1e-12, x, y, z (kpc)]`

**Population_Details columns**: `[index, B0, P0, chi0, B_final, P_final, chi_final, age(Myr), rho_DM, v0_DM, x, y, z, M_NS, R_NS, view_angle, doppler_shift]`